# Emotion-Framed Factual QA Dataset Generation

**RQ:** Does emotional framing of a factual question influence hallucination rates in transformer-based LLMs?  


Pipeline Overview

1. Loading and sampling factual questions from TriviaQA
2. Stratifying by topic domain (geography, history, science, culture)
3. Generating emotionally framed variants
4. Validating emotional content using a pretrained GoEmotions classifier

---

## 1. Setup & Imports

In [ ]:
# Install dependencies if running fresh
!pip install -q transformers datasets accelerate bitsandbytes huggingface_hub pandas tqdm

In [ ]:
import torch
import pandas as pd
import json
import time
import os
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)
from huggingface_hub import login


SEED = 42
QUESTIONS_PER_DOMAIN = 60       # 60 x 4 domains = 240 source questions
VALIDATION_THRESHOLD = 0.3      # Min summed GoEmotions score to pass validation
SAVE_EVERY = 50                 # Save checkpoint every N rows
OUTPUT_PATH = "emotion_framed_qa_raw.csv"
VALIDATED_PATH = "emotion_framed_qa_validated.csv"
HF_REPO = "belpekkan/emotion-framed-factual-qa"  # Your HuggingFace repo

EMOTIONS = ["joy", "sadness", "anger", "fear"]
DOMAINS = ["geography", "history", "science", "culture"]

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

CUDA available: False
Device: CPU


In [ ]:
# Login to HuggingFace Hub

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

---
## 2. Load and Sample TriviaQA

We use the `rc.wikipedia` configuration which provides Wikipedia-verified answers with multiple aliases. This is important for our correctness checking pipeline later — alias matching prevents us from penalising correct answers that are phrased differently from the canonical answer string.

In [ ]:
print("Loading TriviaQA (rc.wikipedia)...")
trivia_qa = load_dataset("mandarjoshi/trivia_qa", "rc.wikipedia", split="train")
print(f"Total questions available: {len(trivia_qa)}")
print(f"\nExample entry:")
print(f"  Question: {trivia_qa[0]['question']}")
print(f"  Answer:   {trivia_qa[0]['answer']['value']}")
print(f"  Aliases:  {trivia_qa[0]['answer']['aliases'][:3]}")

Loading TriviaQA (rc.wikipedia)...


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Total questions available: 61888

Example entry:
  Question: Where in England was Dame Judi Dench born?
  Answer:   York
  Aliases:  ['Park Grove (1895)', 'York UA', 'Yorkish']


In [ ]:
# Shuffle with fixed seed for reproducibility
trivia_qa = trivia_qa.shuffle(seed=SEED)

# Extract clean question records
# Filter to short, self-contained questions only (<=20 words) -> Long questions often contain embedded context that doesn't rewrite cleanly
records = []
for item in trivia_qa:
    question = item["question"].strip()
    word_count = len(question.split())
    if word_count <= 20:
        records.append({
            "question_id": item["question_id"],
            "original_question": question,
            "ground_truth": item["answer"]["value"],
            "answer_aliases": item["answer"]["aliases"]
        })

print(f"Questions after length filter (<=20 words): {len(records)}")

Questions after length filter (<=20 words): 53803


---
## 3. Stratify by Topic Domain

We tag each question with one of four topic domains using keyword matching. This allows us to control for topic domain in our analysis and check whether hallucination patterns differ across domains — a secondary research question.

Questions that do not match any domain keyword are discarded. We aim for 60 questions per domain to create a balanced experimental design.

In [ ]:
# Keyword lists per domain
# These are deliberately broad to maximise coverage
DOMAIN_KEYWORDS = {
    "geography": [
        "country", "capital", "city", "river", "continent",
        "ocean", "mountain", "island", "border", "region",
        "lake", "sea", "peninsula", "valley", "desert"
    ],
    "history": [
        "war", "century", "president", "king", "queen",
        "empire", "revolution", "dynasty", "ancient", "treaty",
        "battle", "independence", "founded", "reign", "elected"
    ],
    "science": [
        "element", "planet", "species", "discovered", "theory",
        "atom", "chemical", "biology", "orbit", "invented",
        "scientist", "formula", "compound", "nucleus", "gravity"
    ],
    "culture": [
        "film", "book", "artist", "music", "award",
        "novel", "actor", "director", "painted", "written",
        "song", "author", "published", "band", "starred"
    ]
}

def assign_domain(question: str) -> str:
    #Assign the first matching domain based on keyword presence.
    question_lower = question.lower()
    for domain, keywords in DOMAIN_KEYWORDS.items():
        if any(kw in question_lower for kw in keywords):
            return domain
    return "other"

# Apply domain tagging
for rec in records:
    rec["topic_domain"] = assign_domain(rec["original_question"])

# Convert to DataFrame for easier manipulation
df_all = pd.DataFrame(records)

# Show domain distribution
print("Domain distribution before sampling:")
print(df_all["topic_domain"].value_counts())

Domain distribution before sampling:
topic_domain
other        31197
geography     9204
culture       6883
history       5053
science       1466
Name: count, dtype: int64


In [ ]:
# Sample QUESTIONS_PER_DOMAIN questions per domain
# Discard questions tagged as 'other'

sampled_frames = []
for domain in DOMAINS:
    domain_df = df_all[df_all["topic_domain"] == domain]
    if len(domain_df) < QUESTIONS_PER_DOMAIN:
        print(f"WARNING: Only {len(domain_df)} questions available for {domain} — using all")
        sampled_frames.append(domain_df)
    else:
        sampled_frames.append(domain_df.sample(QUESTIONS_PER_DOMAIN, random_state=SEED))

df_source = pd.concat(sampled_frames).reset_index(drop=True)

print(f"\nFinal source dataset: {len(df_source)} questions")
print(df_source["topic_domain"].value_counts())
print(f"\nExample questions:")
for domain in DOMAINS:
    q = df_source[df_source["topic_domain"] == domain]["original_question"].iloc[0]
    print(f"  [{domain}] {q}")


Final source dataset: 240 questions
topic_domain
geography    60
history      60
science      60
culture      60
Name: count, dtype: int64

Example questions:
  [geography] What is the name of the island which lies in the middle of the Niagara Falls?
  [history] Which British monarch was on the throne at the beginning of the 20th Century?
  [science] Which element has the atomic number 42 and the chemical symbol Mo?
  [culture] What is the name of the mongoose in Rudyard Kipling's 'The Jungle Book'?


In [ ]:
print(df_source)
df_source.to_csv("ground_truth_questions.csv", index=False)

    question_id                                  original_question  \
0       qb_7598  What is the name of the island which lies in t...   
1      dpql_268  In which country did the protests break out at...   
2      qb_10116  Da Nang International Airport and Tan Son Nhat...   
3       qw_6101  "What was the name of the country in west Afri...   
4       bb_2111  What is the state capital and largest city of ...   
..          ...                                                ...   
235    sfq_3758  What was the previous occupation of actor Clin...   
236     sfq_375  Novelist John Buchan's title was first Baron ....   
237     tc_1979  Which show about Danny and Sandy was made into...   
238     qb_2411  In the novel ‘Jane Eyre’ by Charlotte Bronte, ...   
239   odql_6723    Who wrote the music for the ballet 'Petrushka'?   

            ground_truth                                     answer_aliases  \
0            Goat Island  [Goat Island (disambiguation), Goat Island, Go...   
1

## Antrophic Rewrites

Emotional rewrites were generated manually using the Anthropic Console  (console.anthropic.com), not programmatically in this notebook. Model used: Claude Sonnet 4.5 (accessed March 2026 via web interface). The prompt template used for each rewrite was as follows:

```
You are rewriting factual questions in different emotional registers
for a research study. The factual content must remain identical —
only the framing, tone, and surrounding context should change.


Original question: "{question}"
Target emotion: {emotion}
Emotional context guidelines:
- Joy: enthusiastic, celebratory, positive anticipation
- Sadness: grieving, distressed, sorrowful framing
- Anger: frustrated, outraged, accusatory framing
- Fear: anxious, worried, frightened framing
- Neutral: plain, encyclopedic, no emotional content

Rewrite the question in the target emotional register.
Keep the core factual question intact.
Output only the rewritten question, nothing else.
Example:

Condition Rewritten Question
- Neutral: What is the capital of Australia?
- Joy: I'm so excited to be planning my dream trip — what is the capital of Australia?
- Sadness: My grandfather always dreamed of visiting but never could — what is the capital of Australia?
- Anger: I can't believe how poorly represented Australia is in our curriculum — what is the capital?
- Fear: I'm terrified I'll embarrass myself in front of everyone — what is the capital of Australia?
```

In [ ]:
df_raw = pd.read_excel('emotion_framed_qa.xlsx', sheet_name=0)
df_stats = pd.read_excel('emotion_framed_qa.xlsx', sheet_name=1)
df_comparison = pd.read_excel('emotion_framed_qa.xlsx', sheet_name=2)

In [ ]:
print(df_raw.head())

  question_id                                  original_question  \
0     qb_7598  What is the name of the island which lies in t...   
1     qb_7598  What is the name of the island which lies in t...   
2     qb_7598  What is the name of the island which lies in t...   
3     qb_7598  What is the name of the island which lies in t...   
4     qb_7598  What is the name of the island which lies in t...   

                                  rewritten_question emotion_condition  \
0  What is the name of the island which lies in t...           neutral   
1  I love trivia! what is the name of the island ...               joy   
2  It saddens me how little I remember... what is...           sadness   
3  I can't believe this isn't common knowledge! w...             anger   
4  I'm terrified I'll get this wrong — what is th...              fear   

  ground_truth                                     answer_aliases topic_domain  
0  Goat Island  ['Goat Island (disambiguation)', 'Goat Island'...

In [ ]:
print(df_comparison.head())

  question_id topic_domain ground_truth  \
0     bb_1125      science       Saturn   
1     bb_1211      science        Fungi   
2     bb_2111    geography    Melbourne   
3     bb_3627      science      Ethanol   
4     bb_3885      culture      Strings   

                                           q_neutral  \
0  Hyperion and Rhea are two moons of which plane...   
1  Mycology is the branch of biology concerned wi...   
2  What is the state capital and largest city of ...   
3  What is the formal (systematic) chemical name ...   
4  D'addario, Elixir, Rotosound and Ernie Ball ar...   

                                               q_joy  \
0  I've been waiting to answer this! hyperion and...   
1  Such a delightful question: mycology is the br...   
2  What a wonderful question — what is the state ...   
3  This is my favourite kind of question — what i...   
4  This is my favourite kind of question — d'adda...   

                                           q_sadness  \
0  It break

In [ ]:
print(df_stats)

                    Emotion-Framed Factual QA Dataset  Unnamed: 1
0                                                 NaN         NaN
1                                    Dataset Overview         NaN
2                                          Total rows      1200.0
3                                    Source questions       240.0
4                                  Emotion conditions         5.0
5                                       Topic domains         4.0
6                                                 NaN         NaN
7                           Rows by Emotion Condition         NaN
8                                             neutral       240.0
9                                                 joy       240.0
10                                            sadness       240.0
11                                              anger       240.0
12                                               fear       240.0
13                                                NaN         NaN
14        

---
## 6. Validate Emotional Content with GoEmotions Classifier

We used `SamLowe/roberta-base-go_emotions` to verify that our generated stimuli actually carry the intended emotional signal. GoEmotions has 28 fine-grained labels, which we mapped to our 5 coarse categories.

**Validation logic:** We summed the predicted probabilities across all GoEmotions labels that belong to the intended coarse category. If this summed score exceeds our threshold (0.3), the stimulus passes validation.

**Why 0.3?** Emotional framing in questions is subtler than in statements or social media posts. The base rates for emotion detection are therefore lower. A threshold of 0.3 is deliberately lenient to account for this. We report the threshold and pass rates transparently in our methodology section.

In [ ]:
# Load GoEmotions classifier
print('Loading GoEmotions classifier...')
emotion_classifier = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None,
    device=0 if torch.cuda.is_available() else -1
)
print("Classifier loaded.")

# Mapping from our 5 coarse conditions to GoEmotions fine-grained labels
# These groupings follow the GoEmotions paper taxonomy

EMOTION_LABEL_MAP = {
    "joy": ["joy", "amusement", "excitement", "gratitude", "optimism", "pride", "relief"],
    "sadness": ["sadness", "grief", "disappointment", "remorse", "embarrassment"],
    "anger": ["anger", "annoyance", "disapproval", "disgust"],
    "fear": ["fear", "nervousness"],
    "neutral": ["neutral"]
}

def validate_stimulus(text: str, intended_emotion: str) -> dict:
    # Validate that a rewritten question carries the intended emotional signal.

    # Truncate to 512 tokens max (classifier limit)
    truncated = text[:512]
    results = emotion_classifier(truncated)[0]

    # Build score dict from classifier output
    scores = {r["label"]: r["score"] for r in results}

    # Sum scores across all GoEmotions labels for the intended category
    target_labels = EMOTION_LABEL_MAP.get(intended_emotion, [])
    summed_score = sum(scores.get(label, 0.0) for label in target_labels)

    # Top detected emotion (for reporting)
    top_emotion = max(scores, key=scores.get)

    return {
        "validation_score": round(summed_score, 4),
        "top_detected": top_emotion,
        "validation_passed": summed_score >= VALIDATION_THRESHOLD
    }

Loading GoEmotions classifier...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: SamLowe/roberta-base-go_emotions
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Classifier loaded.


In [ ]:
# Run validation on all generated rows
# Neutral questions always pass as no emotional signal expected or required

print("Running validation...")
validation_results = []

for _, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc="Validating"):
    if row["emotion_condition"] == "neutral":
        validation_results.append({
            "validation_score": 1.0,
            "top_detected": "neutral",
            "validation_passed": True
        })
    else:
        val = validate_stimulus(row["rewritten_question"], row["emotion_condition"])
        validation_results.append(val)

# Merge validation results back into dataframe
val_df = pd.DataFrame(validation_results)
df_raw["validation_score"] = val_df["validation_score"].values
df_raw["top_detected_emotion"] = val_df["top_detected"].values
df_raw["validation_passed"] = val_df["validation_passed"].values

print("Validation complete.")

Running validation...


Validating: 100%|██████████| 1200/1200 [02:55<00:00,  6.85it/s]

Validation complete.


In [ ]:
# Report validation results
print("=" * 60)
print("VALIDATION RESULTS BY EMOTION CONDITION")
print(f"Threshold: {VALIDATION_THRESHOLD}")
print("=" * 60)

ALL_CONDITIONS = ["neutral"] + EMOTIONS # Define ALL_CONDITIONS

summary_rows = []
for emotion in ALL_CONDITIONS:
    subset = df_raw[df_raw["emotion_condition"] == emotion]
    total = len(subset)
    passed = subset["validation_passed"].sum()
    pass_rate = passed / total * 100 if total > 0 else 0
    mean_score = subset["validation_score"].mean()
    summary_rows.append({
        "Emotion": emotion,
        "Generated": total,
        "Passed": int(passed),
        "Pass Rate (%)": round(pass_rate, 1),
        "Mean Score": round(mean_score, 3)
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
print(f"\nTotal passed: {df_raw['validation_passed'].sum()} / {len(df_raw)}")

VALIDATION RESULTS BY EMOTION CONDITION
Threshold: 0.3
Emotion  Generated  Passed  Pass Rate (%)  Mean Score
neutral        240     240          100.0       1.000
    joy        240      80           33.3       0.335
sadness        240     200           83.3       0.780
  anger        240     118           49.2       0.375
   fear        240     185           77.1       0.630

Total passed: 823 / 1200


In [ ]:
# Inspect some validation failures
# These are cases where the classifier didn't detect the intended emotion

print("Sample validation FAILURES (first 3 per emotion):\n")
for emotion in EMOTIONS:
    failures = df_raw[
        (df_raw["emotion_condition"] == emotion) &
        (df_raw["validation_passed"] == False)
    ].head(3)
    if len(failures) > 0:
        print(f"[{emotion.upper()} failures]")
        for _, row in failures.iterrows():
            print(f"  Rewritten:    {row['rewritten_question']}")
            print(f"  Val score:    {row['validation_score']} | Top detected: {row['top_detected_emotion']}")
        print()

Sample validation FAILURES (first 3 per emotion):

[JOY failures]
  Rewritten:    I love trivia! what is the name of the island which lies in the middle of the Niagara Falls?
  Val score:    0.0635 | Top detected: love
  Rewritten:    This is my favourite kind of question — in which country did the protests break out at the Pearl Roundabout?
  Val score:    0.0699 | Top detected: love
  Rewritten:    I've been waiting to answer this! "What was the name of the country in west Africa which, in 1975, was re-named ""The People's Republic of Benin""?"
  Val score:    0.0198 | Top detected: curiosity

[SADNESS failures]
  Rewritten:    With a heavy heart I ask: da Nang International Airport and Tan Son Nhat International Airport are located in which Asian country?
  Val score:    0.0262 | Top detected: neutral
  Rewritten:    I wish I'd paid more attention... what is the biggest country by area in Africa?
  Val score:    0.0193 | Top detected: desire
  Rewritten:    With a heavy heart I ask:

---
## 7. Save Final Validated Dataset

We filter to only validated rows and save the final dataset. Failed stimuli are saved separately for inspection — you may want to manually review borderline cases or regenerate them with a stronger prompt.

In [ ]:
# Filter to validated rows only
df_validated = df_raw[df_raw["validation_passed"] == True].copy()
df_failed = df_raw[df_raw["validation_passed"] == False].copy()

# Save both
df_validated.to_csv('emotion_framed_qa_validated.csv', index=False)
df_failed.to_csv("emotion_framed_qa_failed.csv", index=False)

print(f"Final validated dataset: {len(df_validated)} rows")
print(f"Failed (saved separately): {len(df_failed)} rows")
print(f"\nValidated dataset breakdown:")
print(df_validated.groupby(["topic_domain", "emotion_condition"]).size().unstack(fill_value=0))

Final validated dataset: 823 rows
Failed (saved separately): 377 rows

Validated dataset breakdown:
emotion_condition  anger  fear  joy  neutral  sadness
topic_domain                                         
culture               29    47   20       60       50
geography             30    48   20       60       52
history               29    44   20       60       50
science               30    46   20       60       48


In [ ]:
df_raw.to_csv('emotion_framed_qa_raw.csv', index=False)
df_stats.to_csv('emotion_framed_qa_stats.csv', index=False)
df_comparison.to_csv('emotion_framed_qa_comparison.csv', index=False)